# Solution 2.2.5 — Merging & Combining Datasets

### Path Setup

In [ ]:
import os
import numpy as np
import pandas as pd

features_path = '../../data/20_processed/datania_households_features.csv'
df = pd.read_csv(features_path, dtype={'hh_id': str, 'region_code': str})

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Households:', df.shape)
df[['hh_id', 'region_code', 'province_name', 'district', 'education_code']].head()

---

## Task 1 — A basic left merge with a region lookup

In [ ]:
region_lookup = pd.DataFrame({
    'region_code': ['01', '02', '03', '04', '05', '06'],
    'region_name': ['Eastern', 'Northern', 'Central', 'Southern', 'Western', 'Highland'],
})

merged = pd.merge(df, region_lookup, on='region_code', how='left')
print('Before:', df.shape, '-> After:', merged.shape)
merged[['hh_id', 'region_code', 'region_name']].head(10)

**Questions:**

- Did the row count change? For a left join with a unique key on the right, what should happen to it?
- Some households get a `NaN` `region_name`. Which households, and why (think about what 2.2.3 did to `region_code`)?

**Answers:**

- The row count is unchanged. A left join keeps every left row, and because `region_code` is unique in the lookup each household matches at most one region (many-to-one), so no rows are added.
- Households whose `region_code` became `NaN` in 2.2.3 (the blank cell and the `'99'` sentinel) have no key to match, so their `region_name` is `NaN`.

---

## Task 2 — Join-key hygiene

In [ ]:
df['region_code'] = df['region_code'].astype('string').str.strip()
region_lookup['region_code'] = region_lookup['region_code'].astype('string').str.strip()

print('Missing keys (households):', df['region_code'].isna().sum())
print('Missing keys (lookup):    ', region_lookup['region_code'].isna().sum())

**Questions:**

- How many household rows have a missing join key? After a left join, what value will their `region_name` take?
- Why is fixing the key *at the source* better than dropping the unmatched rows afterwards?

**Answers:**

- A couple of household rows have a missing `region_code`; after a left join their `region_name` is `NaN`. Counting missing keys *before* the merge means the gap is a known fact rather than a surprise in the final table.
- Fixing the key at the source preserves the household (you keep its income, size, etc.) and addresses the real problem — the missing geography — instead of silently deleting records and biasing later totals.

---

## Task 3 — Choose the join type, then audit with `indicator`

In [ ]:
merged = pd.merge(df, region_lookup, on='region_code', how='left', indicator=True)
print(merged['_merge'].value_counts())

In [ ]:
inner = pd.merge(df, region_lookup, on='region_code', how='inner')
outer = pd.merge(df, region_lookup, on='region_code', how='outer')
print('inner:', inner.shape[0], '| left:', merged.shape[0], '| outer:', outer.shape[0])

**Questions:**

- What does each `left_only` row represent here? Why is that group worth reporting back to the data team?
- The inner join has fewer rows than the left join. Which households did it drop?

**Answers:**

- A `left_only` row is a household whose `region_code` is missing, so it matched no region. That group is worth reporting because the geography *should* exist — it points to a collection gap to fix at the source.
- The inner join drops exactly those unmatched households (the ones with a missing `region_code`), which is why it has fewer rows than the left join.

---

## Task 4 — Cardinality: row explosions and `validate`

In [ ]:
bad_lookup = pd.DataFrame({
    'region_code': ['01', '01', '02'],
    'region_name': ['Eastern', 'Eastern (dup)', 'Northern'],
})

exploded = pd.merge(df, bad_lookup, on='region_code', how='left')
print('Rows before:', len(df), '-> after bad merge:', len(exploded))

In [ ]:
try:
    pd.merge(df, bad_lookup, on='region_code', how='left', validate='many_to_one')
except Exception as e:
    print(type(e).__name__, '->', e)

**Questions:**

- Why did the row count grow? Which households were duplicated?
- What do `'one_to_one'`, `'one_to_many'`, and `'many_to_one'` each promise? Which fits a household -> region lookup?

**Answers:**

- Region `'01'` appears twice in `bad_lookup`, so every `'01'` household matches both rows and is duplicated — the row count grows.
- `'one_to_one'`: keys unique on both sides. `'one_to_many'`: unique on the left, repeated on the right. `'many_to_one'`: repeated on the left, unique on the right — that is the right promise for a household -> region lookup, and it raises `MergeError` because the lookup key is not unique.

---

## Task 5 — Merge on differently-named keys

In [ ]:
education_lookup = pd.DataFrame({
    'code': [1, 2, 3, 4],
    'education_label_full': ['No schooling', 'Primary', 'Secondary', 'Tertiary'],
})

merged_edu = pd.merge(
    df,
    education_lookup,
    left_on='education_code',
    right_on='code',
    how='left',
)
merged_edu[['hh_id', 'education_code', 'education_label_full']].head()

**Question:** `education_code` is a float (it carries `NaN`) while `code` is an integer. The merge still matches — why? When would a dtype mismatch silently produce *no* matches instead?

**Answer:** pandas upcasts the two numeric keys to a common type, so the integer `1` and the float `1.0` compare equal and match. A mismatch produces *no* matches when the types are not comparable — most commonly a string `'01'` on one side and an integer `1` on the other, where `'01' != 1`. That is exactly why the hygiene step (`astype`) matters.

---

## Task 6 — Many-to-many: a warning

In [ ]:
roster = pd.DataFrame({'hh_id': [1, 1], 'person': ['A', 'B']})
visits = pd.DataFrame({'hh_id': [1, 1], 'visit': ['V1', 'V2']})

mm = pd.merge(roster, visits, on='hh_id')
print(len(mm), 'rows from two 2-row tables')
mm

**Questions:**

- Why did two people and two visits become four rows?
- What compound key would make each row identify exactly one record?

**Answers:**

- Both sides repeat `hh_id = 1`, so pandas forms every combination: person A x visit V1, A x V2, B x V1, B x V2 — four rows.
- A compound key that is unique per record fixes it. Give each person a `person_id` and each visit a `visit_id`, then merge on the columns that actually identify one row (e.g. `hh_id` + `person_id`).

---

## Task 7 — Post-merge validation, then save the final table

In [ ]:
final = pd.merge(df, region_lookup, on='region_code', how='left')

print('Before:', len(df), '| After:', len(final))
print('Duplicate hh_id:', final['hh_id'].duplicated().sum())
print('Unmatched region:', round(final['region_name'].isna().mean(), 3))

In [ ]:
final = final.reset_index(drop=True)
out_path = '../../data/20_processed/datania_households_merged.csv'

final.to_csv(out_path, index=False)
print('Saved:', out_path, '|', final.shape)

The row count is stable, `hh_id` is still unique, and the unmatched rate equals the share of households with a missing `region_code` — exactly what we expected from Tasks 2-3.

---

## Task 8 — Appending waves with `concat()`

In [ ]:
wave1 = df[['hh_id', 'region_code', 'income_dkw']].head(3).assign(wave='2024')
wave2 = pd.DataFrame({
    'hh_id': ['HH0101', 'HH0102'],
    'region_code': ['01', '02'],
    'income_dkw': [62000.0, 47000.0],
}).assign(wave='2025')

combined = pd.concat([wave1, wave2], ignore_index=True)
combined

**Questions:**

- Why add a `wave` column before stacking?
- `concat()` aligns on column names and fills gaps with `NaN` without warning. What would happen if `wave2` had `income` instead of `income_dkw`? How do you guard against that schema drift?

**Answers:**

- The `wave` column records provenance: once rows are stacked you can no longer tell which survey round a record came from, so you add it *before* concatenating.
- If `wave2` used `income` instead of `income_dkw`, `concat()` would create *both* columns and fill each half with `NaN` — silent schema drift. Guard against it by confirming every input shares the same column names and dtypes before stacking (e.g. compare `wave1.columns` with `wave2.columns`).